# Colab quickstart for 6.S058 trajectory project

Use **Runtime → Change runtime type → GPU**.

**Cursor / VS Code + Colab:** If you connect this notebook to a Colab runtime from your editor, training runs on Google’s GPU while the repo stays on your machine via the extension’s file sync—you do **not** need to copy the project or SportsMOT to Google Drive unless you want an extra backup.

This notebook avoids reinstalling PyTorch because Colab already ships a CUDA-enabled build.

## 1. Check GPU

In [1]:
import torch

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))


## 2. Project directory (local sync)

With the Colab-from-local workflow, `cwd` is usually your repo or `notebooks/`. The next cell resolves `PROJECT_DIR` to the folder that contains `pyproject.toml`.

If you instead use **colab.research.google.com** with no sync, clone into `/content/project` there and set `PROJECT_DIR = Path("/content/project")` manually.

In [2]:
from pathlib import Path

_cwd = Path.cwd()
if (_cwd / "pyproject.toml").exists():
    PROJECT_DIR = _cwd
elif (_cwd.parent / "pyproject.toml").exists():
    PROJECT_DIR = _cwd.parent
else:
    PROJECT_DIR = _cwd

# Colab-in-browser only: clone here and uncomment to override.
# !git clone https://github.com/<you>/<repo>.git /content/project
# PROJECT_DIR = Path("/content/project")

print("PROJECT_DIR =", PROJECT_DIR.resolve())
assert (PROJECT_DIR / "pyproject.toml").is_file(), (
    "Expected repo root (contains pyproject.toml). Open the notebook from the project or fix PROJECT_DIR."
)


## 3. Optional: Google Drive backup

Skip this if you only use local ↔ Colab sync (artifacts under `results/` stay in your repo on disk).

Mount Drive if you want a **second copy** of metrics/checkpoints in Drive after disconnects.

In [ ]:
# Skip this cell when you do not use Drive.

from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

DRIVE_RESULTS = Path("/content/drive/MyDrive/6S058-ballcond-results")
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
print("Drive backup folder:", DRIVE_RESULTS)


## 4. Install dependencies

`requirements-colab.txt` excludes PyTorch so Colab keeps its CUDA wheel.

In [ ]:
import os

os.chdir(PROJECT_DIR)
assert (PROJECT_DIR / "pyproject.toml").is_file(), "PROJECT_DIR must be the repo root."

!python -m pip install -q -r requirements-colab.txt
!python -m pip install -q -e .

import ballcond
print('ballcond:', ballcond.__version__)


## 5. Run smoke tests

In [ ]:
!python -m pytest tests/ -q


## 6. Quick GPU training run

This creates a tiny one-epoch config from `configs/synthetic_ballcond.yaml` and runs the main model. It should complete quickly on a Colab GPU.

In [ ]:
from omegaconf import OmegaConf

cfg = OmegaConf.load('configs/synthetic_ballcond.yaml')
cfg.run_name = 'colab_smoke_ballcond'
cfg.device = 'cuda' if torch.cuda.is_available() else 'cpu'
cfg.data.num_sequences = 40
cfg.train.epochs = 1
cfg.train.batch_size = 64

smoke_cfg = PROJECT_DIR / "colab_smoke_ballcond.yaml"
OmegaConf.save(cfg, str(smoke_cfg))
!python -m ballcond.train --config colab_smoke_ballcond.yaml --out results


## 7. Full synthetic sweep

Run this when the smoke run works. It trains the Kalman, LSTM, symmetric transformer, and ball-conditioned transformer.

In [ ]:
!bash scripts/run_synthetic_sweep.sh


## 8. Optional: copy `results/` to Drive

Skip if you skipped section 3.

In [ ]:
import shutil
from pathlib import Path


def copy_children(src: Path, dst: Path) -> None:
    if not src.exists():
        return
    dst.mkdir(parents=True, exist_ok=True)
    for child in src.iterdir():
        target = dst / child.name
        if target.exists():
            if target.is_dir():
                shutil.rmtree(target)
            else:
                target.unlink()
        if child.is_dir():
            shutil.copytree(child, target)
        else:
            shutil.copy2(child, target)


drive_results = globals().get("DRIVE_RESULTS")
if drive_results is not None:
    copy_children(PROJECT_DIR / "results", drive_results)
    print("Copied project results/ to", drive_results)
else:
    print("No Drive mount — outputs are in", PROJECT_DIR / "results")
